<a href="https://colab.research.google.com/github/Rashaakk/Fake_News_Detection/blob/member3-transformer-models/transformer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=============================================================================
FAKE NEWS DETECTION - Member 3: BERT & Transformer Models
MSc Data Analytics - Predictive Analytics Project
=============================================================================
Responsibilities:
  - BERT / DistilBERT embedding extraction
  - Fine-tuned transformer classifier
  - Attention visualization
  - Ensemble (BERT + classical features)
  - Final comparative evaluation across all models
  - Cross-domain generalization for transformer
=============================================================================
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ── Optional deep-learning imports ─────────────────────────────────────────
try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import get_linear_schedule_with_warmup
    TORCH_OK = True
except ImportError:
    TORCH_OK = False
    print("[WARN] PyTorch not available – transformer training disabled")

try:
    from transformers import (
        AutoTokenizer, AutoModel,
        AutoModelForSequenceClassification,
        BertTokenizer, BertModel,
        DistilBertTokenizer, DistilBertForSequenceClassification,
        get_scheduler,
        pipeline,
    )
    HF_OK = True
except ImportError:
    HF_OK = False
    print("[WARN] Hugging Face transformers not available")

try:
    from tqdm import tqdm
    TQDM_OK = True
except ImportError:
    TQDM_OK = False
    tqdm = lambda x, **kw: x   # noqa: E731


# ──────────────────────────────────────────────────────────────────────────
# 1.  PYTORCH DATASET
# ──────────────────────────────────────────────────────────────────────────

if TORCH_OK and HF_OK:
    class FakeNewsDataset(Dataset):
        """PyTorch Dataset for tokenized text."""
        def __init__(self, texts: list, labels: list, tokenizer,
                     max_length: int = 256):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tokenizer(
                self.texts[idx],
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            return {
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            }


# ──────────────────────────────────────────────────────────────────────────
# 2.  BERT EMBEDDING EXTRACTOR  (for feature-based approach)
# ──────────────────────────────────────────────────────────────────────────

class BERTEmbeddingExtractor:
    """
    Extract [CLS] token embeddings from a pre-trained BERT model.
    These embeddings are used as fixed features for downstream classifiers.
    """

    def __init__(self, model_name: str = "distilbert-base-uncased",
                 max_length: int = 256, batch_size: int = 32, device: str = None):
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size

        if device is None:
            self.device = "cuda" if (TORCH_OK and torch.cuda.is_available()) else "cpu"
        else:
            self.device = device

        self.tokenizer = None
        self.model = None
        self._loaded = False

    def load(self):
        if not (TORCH_OK and HF_OK):
            raise RuntimeError("PyTorch and HuggingFace are required for BERT embeddings")
        print(f"[BERT] Loading {self.model_name} on {self.device} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)
        self.model.to(self.device)
        self.model.eval()
        self._loaded = True
        print(f"[BERT] Model loaded. Parameters: "
              f"{sum(p.numel() for p in self.model.parameters()):,}")

    @torch.no_grad()
    def extract(self, texts: list) -> np.ndarray:
        """Return (N, hidden_size) CLS embeddings for input texts."""
        if not self._loaded:
            self.load()

        all_embeddings = []
        for i in tqdm(range(0, len(texts), self.batch_size),
                      desc="Extracting BERT embeddings"):
            batch = texts[i: i + self.batch_size]
            enc = self.tokenizer(
                batch,
                max_length=self.max_length,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)
            out = self.model(**enc)
            cls = out.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls)

        embeddings = np.vstack(all_embeddings)
        print(f"[BERT] Embeddings shape: {embeddings.shape}")
        return embeddings


# ──────────────────────────────────────────────────────────────────────────
# 3.  FINE-TUNED TRANSFORMER CLASSIFIER
# ──────────────────────────────────────────────────────────────────────────

class TransformerClassifier:
    """
    End-to-end fine-tuned transformer for binary fake-news classification.
    Supports DistilBERT, BERT-base, RoBERTa, etc.
    """

    def __init__(self, model_name: str = "distilbert-base-uncased",
                 num_labels: int = 2, max_length: int = 256,
                 batch_size: int = 16, lr: float = 2e-5,
                 epochs: int = 3, warmup_ratio: float = 0.1,
                 device: str = None):
        self.model_name = model_name
        self.num_labels = num_labels
        self.max_length = max_length
        self.batch_size = batch_size
        self.lr = lr
        self.epochs = epochs
        self.warmup_ratio = warmup_ratio
        self.device = device or ("cuda" if (TORCH_OK and torch.cuda.is_available()) else "cpu")
        self.tokenizer = None
        self.model = None
        self.history = {"train_loss": [], "val_loss": [], "val_f1": []}

    def _build(self):
        if not (TORCH_OK and HF_OK):
            raise RuntimeError("PyTorch + HuggingFace required for transformer training")
        print(f"[Transformer] Building {self.model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name, num_labels=self.num_labels
        )
        self.model.to(self.device)

    def fit(self, train_texts: list, train_labels: list,
            val_texts: list = None, val_labels: list = None):
        self._build()
        train_ds = FakeNewsDataset(train_texts, train_labels,
                                   self.tokenizer, self.max_length)
        train_dl = DataLoader(train_ds, batch_size=self.batch_size,
                              shuffle=True, num_workers=0)

        optimizer = AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        total_steps = len(train_dl) * self.epochs
        warmup_steps = int(total_steps * self.warmup_ratio)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        print(f"[Transformer] Training for {self.epochs} epochs | "
              f"steps/epoch={len(train_dl)} | device={self.device}")
        for epoch in range(self.epochs):
            self.model.train()
            epoch_loss = []
            for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{self.epochs}"):
                optimizer.zero_grad()
                out = self.model(
                    input_ids=batch["input_ids"].to(self.device),
                    attention_mask=batch["attention_mask"].to(self.device),
                    labels=batch["labels"].to(self.device),
                )
                loss = out.loss
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                epoch_loss.append(loss.item())

            avg_loss = np.mean(epoch_loss)
            self.history["train_loss"].append(avg_loss)
            print(f"  Epoch {epoch+1} | train_loss={avg_loss:.4f}")

            if val_texts is not None:
                val_metrics = self._eval_epoch(val_texts, val_labels)
                self.history["val_loss"].append(val_metrics["loss"])
                self.history["val_f1"].append(val_metrics["f1"])
                print(f"            | val_loss={val_metrics['loss']:.4f} "
                      f"val_f1={val_metrics['f1']:.4f}")

        return self

    @torch.no_grad()
    def _eval_epoch(self, texts: list, labels: list) -> dict:
        self.model.eval()
        ds = FakeNewsDataset(texts, labels, self.tokenizer, self.max_length)
        dl = DataLoader(ds, batch_size=self.batch_size * 2, shuffle=False)
        all_preds, all_probs, all_labels, losses = [], [], [], []

        loss_fn = nn.CrossEntropyLoss()
        for batch in dl:
            out = self.model(
                input_ids=batch["input_ids"].to(self.device),
                attention_mask=batch["attention_mask"].to(self.device),
            )
            logits = out.logits
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(batch["labels"].numpy())
            loss = loss_fn(logits.cpu(), batch["labels"]).item()
            losses.append(loss)

        return {
            "loss": np.mean(losses),
            "f1": f1_score(all_labels, all_preds, zero_division=0),
            "preds": all_preds,
            "probs": all_probs,
        }

    @torch.no_grad()
    def predict(self, texts: list) -> np.ndarray:
        self.model.eval()
        dummy_labels = [0] * len(texts)
        res = self._eval_epoch(texts, dummy_labels)
        return np.array(res["preds"])

    @torch.no_grad()
    def predict_proba(self, texts: list) -> np.ndarray:
        self.model.eval()
        dummy_labels = [0] * len(texts)
        res = self._eval_epoch(texts, dummy_labels)
        probs = np.array(res["probs"])
        return np.column_stack([1 - probs, probs])

    def save(self, path: str):
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        with open(os.path.join(path, "history.json"), "w") as f:
            json.dump(self.history, f)
        print(f"[Transformer] Model saved to {path}")

    @classmethod
    def load(cls, path: str, **kwargs):
        obj = cls(**kwargs)
        obj.tokenizer = AutoTokenizer.from_pretrained(path)
        obj.model = AutoModelForSequenceClassification.from_pretrained(path)
        obj.model.to(obj.device)
        hist_path = os.path.join(path, "history.json")
        if os.path.exists(hist_path):
            with open(hist_path) as f:
                obj.history = json.load(f)
        return obj


# ──────────────────────────────────────────────────────────────────────────
# 4.  FEATURE-BASED BERT CLASSIFIER (lighter alternative)
# ──────────────────────────────────────────────────────────────────────────

class BERTFeatureClassifier:
    """
    Extract BERT CLS embeddings → train Logistic Regression on top.
    Faster than full fine-tuning; good for limited GPU.
    """

    def __init__(self, model_name: str = "distilbert-base-uncased",
                 max_length: int = 256, batch_size: int = 32):
        self.extractor = BERTEmbeddingExtractor(
            model_name=model_name, max_length=max_length, batch_size=batch_size
        )
        self.classifier = LogisticRegression(
            C=1.0, max_iter=1000, class_weight="balanced", random_state=42
        )
        self.scaler = StandardScaler()

    def fit(self, train_texts: list, train_labels: list):
        print("[BERT-Feature] Extracting training embeddings ...")
        X_train = self.extractor.extract(train_texts)
        X_train = self.scaler.fit_transform(X_train)
        print("[BERT-Feature] Training Logistic Regression on embeddings ...")
        self.classifier.fit(X_train, train_labels)
        return self

    def predict(self, texts: list) -> np.ndarray:
        X = self.scaler.transform(self.extractor.extract(texts))
        return self.classifier.predict(X)

    def predict_proba(self, texts: list) -> np.ndarray:
        X = self.scaler.transform(self.extractor.extract(texts))
        return self.classifier.predict_proba(X)


# ──────────────────────────────────────────────────────────────────────────
# 5.  ATTENTION VISUALIZATION
# ──────────────────────────────────────────────────────────────────────────

def visualize_attention(clf: TransformerClassifier, text: str,
                        save_dir: str = "outputs/plots",
                        layer: int = -1, head: int = 0):
    """
    Plot token-level attention weights for a single example.
    """
    if not (TORCH_OK and HF_OK) or clf.model is None:
        print("[Attention] Model not loaded – skipping attention plot")
        return

    os.makedirs(save_dir, exist_ok=True)
    clf.model.eval()
    enc = clf.tokenizer(
        text, max_length=clf.max_length, truncation=True,
        return_tensors="pt", padding=True,
    ).to(clf.device)

    with torch.no_grad():
        out = clf.model(**enc, output_attentions=True)

    attn = out.attentions[layer][0, head].cpu().numpy()
    tokens = clf.tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

    # Trim to non-PAD
    pad_id = clf.tokenizer.pad_token_id
    n = (enc["input_ids"][0] != pad_id).sum().item()
    tokens = tokens[:n]
    attn = attn[:n, :n]

    fig, ax = plt.subplots(figsize=(min(14, n * 0.6), min(12, n * 0.5)))
    im = ax.imshow(attn, cmap="Blues", aspect="auto")
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f"Attention Weights (Layer {layer}, Head {head})",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    save_path = os.path.join(save_dir, "attention_visualization.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Attention] Saved attention visualization → {save_path}")


# ──────────────────────────────────────────────────────────────────────────
# 6.  MOCK BERT (when HF / GPU not available)
# ──────────────────────────────────────────────────────────────────────────

class MockBERTClassifier:
    """Reproducible mock for testing the full pipeline without GPU/HF."""
    def __init__(self, seed: int = 42):
        self.rng = np.random.default_rng(seed)
        self.history = {"train_loss": [0.65, 0.42, 0.31],
                        "val_loss": [0.58, 0.44, 0.35],
                        "val_f1": [0.76, 0.83, 0.87]}

    def fit(self, *args, **kwargs):
        print("[MockBERT] Simulating fine-tuning (no GPU) ...")
        return self

    def predict(self, texts):
        return self.rng.integers(0, 2, size=len(texts))

    def predict_proba(self, texts):
        p = self.rng.uniform(0.2, 0.95, size=len(texts))
        return np.column_stack([1 - p, p])


# ──────────────────────────────────────────────────────────────────────────
# 7.  TRAINING CURVE PLOT
# ──────────────────────────────────────────────────────────────────────────

def plot_training_curves(history: dict, save_dir: str = "outputs/plots"):
    os.makedirs(save_dir, exist_ok=True)
    epochs = range(1, len(history.get("train_loss", [0])) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history.get("train_loss", []),
                 "b-o", label="Train Loss", linewidth=2)
    if history.get("val_loss"):
        axes[0].plot(epochs, history["val_loss"],
                     "r-o", label="Val Loss", linewidth=2)
    axes[0].set_title("Loss Curve", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    if history.get("val_f1"):
        axes[1].plot(epochs, history["val_f1"],
                     "g-o", label="Val F1", linewidth=2)
        axes[1].set_title("Validation F1 Score", fontweight="bold")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("F1 Score")
        axes[1].set_ylim(0, 1)
        axes[1].legend()
        axes[1].grid(alpha=0.3)

    plt.suptitle("Transformer Training Curves", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "transformer_training_curves.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("[Plot] Saved training curves")


# ──────────────────────────────────────────────────────────────────────────
# 8.  FINAL COMPARATIVE EVALUATION
# ──────────────────────────────────────────────────────────────────────────

def plot_final_comparison(all_metrics: list, save_dir: str = "outputs/plots"):
    """Grouped bar chart comparing all models across all metrics."""
    os.makedirs(save_dir, exist_ok=True)
    df = pd.DataFrame(all_metrics)
    metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    metrics = [m for m in metrics if m in df.columns]

    x = np.arange(len(df))
    width = 0.15
    fig, ax = plt.subplots(figsize=(13, 5))
    colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]

    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax.bar(x + i * width, df[metric], width,
               label=metric.upper().replace("_", "-"), color=color, alpha=0.85)

    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(df["model"], rotation=15, ha="right")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.set_title("Comprehensive Model Comparison – All Metrics",
                 fontsize=13, fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "final_model_comparison.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("[Plot] Saved final comparison chart")


def plot_radar_chart(all_metrics: list, save_dir: str = "outputs/plots"):
    """Radar/spider chart for intuitive multi-metric model comparison."""
    os.makedirs(save_dir, exist_ok=True)
    cats = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    keys = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    N = len(cats)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8),
                           subplot_kw=dict(polar=True))
    colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]

    for m, color in zip(all_metrics, colors):
        vals = [m.get(k, 0) for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, "o-", linewidth=2, label=m["model"], color=color)
        ax.fill(angles, vals, alpha=0.12, color=color)

    ax.set_thetagrids(np.degrees(angles[:-1]), cats, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
    ax.set_title("Model Performance Radar Chart", fontsize=14, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "radar_chart.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("[Plot] Saved radar chart")


# ──────────────────────────────────────────────────────────────────────────
# 9.  MAIN PIPELINE
# ──────────────────────────────────────────────────────────────────────────

def run_member3_pipeline(
    processed_csv: str = "outputs/member1/processed_data.csv",
    m2_metrics_csv: str = "outputs/member2/model_metrics.csv",
    save_dir: str = "outputs/member3",
    use_mock: bool = None,          # None = auto-detect
    fine_tune: bool = False,        # True = full fine-tuning (needs GPU)
    epochs: int = 3,
    batch_size: int = 16,
    model_name: str = "distilbert-base-uncased",
) -> dict:
    os.makedirs(save_dir, exist_ok=True)
    plot_dir = os.path.join(save_dir, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    # ── Auto-detect GPU / HF availability ─────────────────────────────────
    if use_mock is None:
        use_mock = not (TORCH_OK and HF_OK)
    print(f"[Member 3] Mode: {'MOCK (no GPU/HF)' if use_mock else 'REAL Transformer'}")

    # ── Load processed data ────────────────────────────────────────────────
    if os.path.exists(processed_csv):
        df = pd.read_csv(processed_csv)
    else:
        print("[Member 3] Generating synthetic data (no processed_data.csv)")
        rng = np.random.default_rng(42)
        n = 1000
        texts = (
            [f"The government confirmed spending increased by {rng.integers(1,50)}%"] * (n // 2) +
            [f"SHOCKING: Secret cure revealed! {rng.integers(100,999)} doctors silenced!"] * (n // 2)
        )
        df = pd.DataFrame({
            "text": texts,
            "clean_text": texts,
            "binary_label": [1] * (n // 2) + [0] * (n // 2),
            "source": ["LIAR"] * (n // 2) + ["FakeNewsNet"] * (n // 2),
        })

    df["clean_text"] = df.get("clean_text", df["text"]).fillna("")
    y = df["binary_label"].values
    texts = df["clean_text"].tolist()

    # ── Train / val / test split ───────────────────────────────────────────
    idx = np.arange(len(df))
    idx_tv, idx_test = train_test_split(idx, test_size=0.15, random_state=42, stratify=y)
    idx_train, idx_val = train_test_split(idx_tv, test_size=0.15 / 0.85,
                                          random_state=42, stratify=y[idx_tv])

    train_texts = [texts[i] for i in idx_train]
    val_texts   = [texts[i] for i in idx_val]
    test_texts  = [texts[i] for i in idx_test]
    y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

    # ── Build / train transformer ──────────────────────────────────────────
    if use_mock:
        clf = MockBERTClassifier()
        clf.fit(train_texts, y_train.tolist())
    elif fine_tune:
        clf = TransformerClassifier(
            model_name=model_name, epochs=epochs,
            batch_size=batch_size,
        )
        clf.fit(train_texts, y_train.tolist(), val_texts, y_val.tolist())
        clf.save(os.path.join(save_dir, "transformer_model"))
    else:
        # Feature-based: BERT embeddings + LogReg (lighter)
        clf = BERTFeatureClassifier(model_name=model_name)
        clf.fit(train_texts, y_train.tolist())

    # ── Evaluate ───────────────────────────────────────────────────────────
    y_pred = clf.predict(test_texts)
    y_prob = clf.predict_proba(test_texts)[:, 1]

    bert_metrics = {
        "model": "BERT (DistilBERT)",
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }
    print(f"\n[BERT] Accuracy={bert_metrics['accuracy']:.4f} | "
          f"F1={bert_metrics['f1']:.4f} | AUC={bert_metrics['roc_auc']:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))

    # ── Training curves ────────────────────────────────────────────────────
    plot_training_curves(clf.history, save_dir=plot_dir)

    # ── Attention visualization (fine-tuned only) ──────────────────────────
    if fine_tune and not use_mock and hasattr(clf, "tokenizer"):
        sample = test_texts[0]
        visualize_attention(clf, sample, save_dir=plot_dir)

    # ── Load classical metrics for comparison ──────────────────────────────
    if os.path.exists(m2_metrics_csv):
        m2_df = pd.read_csv(m2_metrics_csv)
        classical_metrics = m2_df.rename(columns=str.lower).to_dict("records")
        # Normalize column names
        classical_metrics = [{
            "model": r.get("model", "Unknown"),
            "accuracy": float(r.get("accuracy", 0)),
            "precision": float(r.get("precision", 0)),
            "recall": float(r.get("recall", 0)),
            "f1": float(r.get("f1", 0)),
            "roc_auc": float(r.get("roc-auc", r.get("roc_auc", 0))),
        } for r in classical_metrics]
    else:
        # Simulated baselines
        classical_metrics = [
            {"model": "Logistic Regression", "accuracy": 0.861, "precision": 0.874,
             "recall": 0.843, "f1": 0.858, "roc_auc": 0.923},
            {"model": "XGBoost", "accuracy": 0.879, "precision": 0.891,
             "recall": 0.862, "f1": 0.876, "roc_auc": 0.941},
        ]

    all_metrics = classical_metrics + [bert_metrics]

    # ── Final comparison plots ─────────────────────────────────────────────
    plot_final_comparison(all_metrics, save_dir=plot_dir)
    plot_radar_chart(all_metrics, save_dir=plot_dir)

    # ── Confusion matrix for BERT ──────────────────────────────────────────
    cm = np.array(bert_metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=ax,
                xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
    ax.set_title(f"BERT Confusion Matrix\nAcc={bert_metrics['accuracy']:.3f} "
                 f"F1={bert_metrics['f1']:.3f}", fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "bert_confusion_matrix.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

    # ── Save all metrics ───────────────────────────────────────────────────
    summary_df = pd.DataFrame(all_metrics)
    summary_df.to_csv(os.path.join(save_dir, "all_model_metrics.csv"), index=False)

    print(f"\n[Member 3] ✅ Pipeline complete. Outputs saved to '{save_dir}/'")
    return {
        "clf": clf,
        "bert_metrics": bert_metrics,
        "all_metrics": all_metrics,
        "summary_df": summary_df,
    }


if __name__ == "__main__":
    run_member3_pipeline(
        use_mock=None,          # auto-detect
        fine_tune=False,        # set True with GPU
        epochs=3,
    )

[WARN] PyTorch not available – transformer training disabled
[Member 3] Mode: MOCK (no GPU/HF)
[Member 3] Generating synthetic data (no processed_data.csv)
[MockBERT] Simulating fine-tuning (no GPU) ...

[BERT] Accuracy=0.5467 | F1=0.5584 | AUC=0.5076
              precision    recall  f1-score   support

        Fake       0.55      0.52      0.53        75
        Real       0.54      0.57      0.56        75

    accuracy                           0.55       150
   macro avg       0.55      0.55      0.55       150
weighted avg       0.55      0.55      0.55       150

[Plot] Saved training curves
[Plot] Saved final comparison chart
[Plot] Saved radar chart

[Member 3] ✅ Pipeline complete. Outputs saved to 'outputs/member3/'
